In [ ]:
import sys
sys.path.insert(0, '/data/prayoga/MinPrep/')

import warnings
import mercury as mr
import missingno as msno
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, HTML

from minprep_core import (
    missing_values_table,
    preprocess_for_training,
    certain_clean_main,
    MR_main,
    MR_main_regression,
)

warnings.filterwarnings("ignore")
mr.StopExecution._render_traceback_ = lambda self: []

show_code = mr.Checkbox(label="Show code", value=False)
app = mr.App(title="MinPrep", description="Train ML pipeline ", show_code=show_code.value)

In [ ]:
from IPython.display import display, HTML
html_code = '''
<style>
    #iframe-container {
        position: fixed;
        top: 0;
        right: 0;
        width: 18%; /* Adjust the width as needed */
        height: 100%; /* Adjust the height as needed */
        z-index: 9999;
    }
    #iframe-container iframe {
        width: 100%;
        height: 100%;
        border: none;
    }
</style>
<div id="iframe-container">
    <iframe src="http://127.0.0.1:8000/app/certain-prep-explorer-view"></iframe>
</div>
<script>
    document.querySelector("#iframe-container iframe").addEventListener("click", function() {
        window.open(this.src, '_blank');
    });
</script>
'''

display(HTML(html_code))

In [ ]:
data_file = mr.File(label="Upload Dataset", max_file_size="10MB")
df=pd.read_csv('/data/prayoga/MI/SVM/real/data/original/water.csv')

In [ ]:
if data_file.filepath is None:
    mr.Stop()
else:
    df = pd.read_csv(data_file.filepath)
    # print(data_file.filepath)
    df.to_csv("/data/prayoga/MinPrep/shared/data.csv", index=False)


In [ ]:
# ── Data Selection ────────────────────────────────────────────────────────────
x_columns      = mr.MultiSelect(label="Input features", value=list(df.columns)[:-1], choices=list(df.columns))
y_column        = mr.Select(label="Target", value=list(df.columns)[-1], choices=list(df.columns))

# ── Model Configuration ───────────────────────────────────────────────────────
_ = mr.Note("<span style='font-size:20px'>**Model Configuration**</span>")
is_acm          = mr.Checkbox(label="Evaluate ACM", value=True)
verbose         = mr.Checkbox(label="Return Samples Requiring Imputation")
ACM_Threshold   = mr.Slider(value=5, min=0, max=10, label="ACM Threshold", step=1)
eval_metric     = mr.Select(label="Evaluation Metric", value="Accuracy", choices=["Accuracy", "RMSE", "MSE"])
time_limit      = mr.Numeric(label="Running Time Limit (Minutes)", value=1, min=1, max=150)

# ── Additional Settings ───────────────────────────────────────────────────────
_ = mr.Note("<span style='font-size:20px'>**Additional Settings**</span>")
cleaning_function = mr.File(label="Provide Imputation Function (Default Mean)", max_file_size="1MB")

# ── Run ───────────────────────────────────────────────────────────────────────
start_training  = mr.Button(label="Start Training", style="success")
output_dir      = mr.OutputDir()

In [ ]:
if x_columns.value is None or len(x_columns.value) == 0 or y_column.value is None:
    # print("Please select input features and target column")
    mr.Stop()

In [ ]:
mr.Markdown("# Dirty Value Diagnostics")
import matplotlib.pyplot as plt
%matplotlib inline
missing_values_table(df).to_markdown() # Adjust the figure size as needed
msno.matrix(df, figsize=(15, 8))  # Assuming this function generates a visualization
plt.show()

In [ ]:
output_csv = None
missing_data_table = None
cm_passed = None

if start_training.clicked:
    mr.Markdown("# MinPrep Training Results")

    # Step 0: Preprocessing — restrict to selected input features + target,
    # one-hot encode any non-numeric feature columns, split train/test
    X_train, Y_train, X_test, y_test = preprocess_for_training(df, x_columns.value, y_column.value)

    # Step 1: CM check
    # The task is classification or regression
    task_type = 'classification' if eval_metric.value == 'Accuracy' else 'regression'
    cm_results, missing_data_table, cm_passed, _ = certain_clean_main(
        X_train, Y_train, X_test, y_test, task_type=task_type, verbose=verbose.value)

    if cm_passed:
        # CM exists — data is clean, no imputation needed
        output_csv = cm_results
        mr.Markdown("**CM Result: Exists** — No imputation required.")
    else:
        # CM does not exist — run Minimal Repair
        mr.Markdown("**CM Result: Does not Exist** — Running Minimal Repair (MR)...")
        if task_type == 'regression':
            output_csv, missing_data_table = MR_main_regression(X_train, Y_train, X_test, y_test)
        else:
            output_csv, missing_data_table = MR_main(X_train, Y_train, X_test, y_test)

    pd.DataFrame(output_csv).to_csv(
        "/data/prayoga/MinPrep/shared/certain_prep_results.csv", index=False)
    mr.Confetti()

In [ ]:
my_folder = mr.OutputDir()
first_file_name = os.path.join(my_folder.path, "output-file.txt")
with open(first_file_name, "w") as fout:
    fout.write("Hello there!")

In [ ]:
if output_csv is None:
    mr.Stop()
from IPython.display import display, HTML

cm_lookup = cm_results.set_index('Metric')['Value']
cm_result_str = cm_lookup['CM Result']
cm_time = cm_lookup['Running Time (CM)']
score_label = 'Accuracy (CM)' if 'Accuracy (CM)' in cm_lookup.index else 'MSE (CM)'
cm_accuracy = cm_lookup[score_label]

display(HTML(f'''<hr style='border:none;border-top:2px solid #e9ecef;margin:24px 0;'><div style='padding:14px 20px;background:#f8f9fa;border-left:5px solid #2563eb;border-radius:0 8px 8px 0;margin-bottom:16px;'><h3 style='margin:0;color:#2c3e50;font-size:20px;font-weight:600;'>CM Result: {cm_result_str}</h3></div>'''))

cm_boxes = [
    (round(cm_time, 4),     'CM Running Time (s)'),
    (round(cm_accuracy, 4), score_label),
]
html = "<div style='display:flex;flex-wrap:wrap;gap:14px;margin:8px 0 24px 0;'>"
for data, title in cm_boxes:
    html += f"""
    <div style=\'border:1px solid #e0e0e0;border-top:3px solid #2563eb;border-radius:8px;padding:18px 24px;text-align:center;min-width:140px;background:#fff;box-shadow:0 2px 4px rgba(0,0,0,0.06);\'>\n        <div style=\'font-size:20px;color:#6c757d;margin-bottom:8px;text-transform:uppercase;letter-spacing:0.5px;\'>{title}</div>\n        <div style=\'font-size:34px;font-weight:700;color:#212529;\'>{data}</div>\n    </div>"""
html += "</div>"
display(HTML(html))


In [ ]:
if not cm_passed:
    mr_lookup = output_csv.set_index('Metric')['Value']
    display(HTML(f'''<hr style='border:none;border-top:2px solid #e9ecef;margin:24px 0;'><div style='padding:14px 20px;background:#f8f9fa;border-left:5px solid #7c3aed;border-radius:0 8px 8px 0;margin-bottom:16px;'><h3 style='margin:0;color:#2c3e50;font-size:20px;font-weight:600;'>Minimal Imputation Result</h3></div>'''))

    mr_score_label = 'Accuracy (MR)' if 'Accuracy (MR)' in mr_lookup.index else 'MSE (MR)'
    minprep_boxes = [
        (mr_lookup['% Repaired (MR)'],          '% Sample Imputed'),
        (round(mr_lookup['Running Time (MR)'], 4), 'Total Running Time (s)'),
        (round(mr_lookup[mr_score_label], 4),     mr_score_label),
    ]
    html = "<div style='display:flex;flex-wrap:wrap;gap:14px;margin:8px 0 24px 0;'>"
    for data, title in minprep_boxes:
        html += f"""
        <div style=\'border:1px solid #e0e0e0;border-top:3px solid #7c3aed;border-radius:8px;padding:18px 24px;text-align:center;min-width:140px;background:#fff;box-shadow:0 2px 4px rgba(0,0,0,0.06);\'>\n        <div style=\'font-size:20px;color:#6c757d;margin-bottom:8px;text-transform:uppercase;letter-spacing:0.5px;\'>{title}</div>\n        <div style=\'font-size:34px;font-weight:700;color:#212529;\'>{data}</div>\n    </div>"""
    html += "</div>"
    display(HTML(html))
    

In [ ]:
display(HTML(f'''<hr style='border:none;border-top:2px solid #e9ecef;margin:24px 0;'><div style='padding:14px 20px;background:#f8f9fa;border-left:5px solid #d97706;border-radius:0 8px 8px 0;margin-bottom:16px;'><h3 style='margin:0;color:#2c3e50;font-size:20px;font-weight:600;'>AC Result</h3></div>'''))

ac_boxes = [
    ('NA', '% Sample Imputed'),
    ('NA', 'Total Running Time (s)'),
    ('NA', 'Test Accuracy (%)'),
]
html = "<div style='display:flex;flex-wrap:wrap;gap:14px;margin:8px 0 24px 0;'>"
for data, title in ac_boxes:
    html += f"""
    <div style=\'border:1px solid #e0e0e0;border-top:3px solid #d97706;border-radius:8px;padding:18px 24px;text-align:center;min-width:140px;background:#fff;box-shadow:0 2px 4px rgba(0,0,0,0.06);\'>\n        <div style=\'font-size:20px;color:#6c757d;margin-bottom:8px;text-transform:uppercase;letter-spacing:0.5px;\'>{title}</div>\n        <div style=\'font-size:34px;font-weight:700;color:#212529;\'>{data}</div>\n    </div>"""
html += "</div>"
display(HTML(html))


In [ ]:
display(HTML(f'''<hr style='border:none;border-top:2px solid #e9ecef;margin:24px 0;'><div style='padding:14px 20px;background:#f8f9fa;border-left:5px solid #059669;border-radius:0 8px 8px 0;margin-bottom:16px;'><h3 style='margin:0;color:#2c3e50;font-size:20px;font-weight:600;'>Full Imputation Result</h3></div>'''))

fi_boxes = [
    ('NA', '% Sample Imputed'),
    ('NA', 'Total Running Time (s)'),
    ('NA', 'Test Accuracy (%)'),
]
html = "<div style='display:flex;flex-wrap:wrap;gap:14px;margin:8px 0 24px 0;'>"
for data, title in fi_boxes:
    html += f"""
    <div style=\'border:1px solid #e0e0e0;border-top:3px solid #059669;border-radius:8px;padding:18px 24px;text-align:center;min-width:140px;background:#fff;box-shadow:0 2px 4px rgba(0,0,0,0.06);\'>\n        <div style=\'font-size:20px;color:#6c757d;margin-bottom:8px;text-transform:uppercase;letter-spacing:0.5px;\'>{title}</div>\n        <div style=\'font-size:34px;font-weight:700;color:#212529;\'>{data}</div>\n    </div>"""
html += "</div>"
display(HTML(html))


In [ ]:
#add a line
mr.Markdown(" ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
mr.Markdown("# Minimal Imputation")

In [ ]:
if missing_data_table is not None and len(missing_data_table) > 0:
    display(missing_data_table.sample(min(5, len(missing_data_table))))